**Навигация по уроку**

1. [Telegram бот c ChatGPT на борту. Знакомство с  ChatGPT](https://colab.research.google.com/drive/1tqXt0XNstMb2TeJ8Za9S2xQA6ipjyMPk)
2. [Подготовка данных для обучения chatGPT методом Search-Ask (Практика)](https://colab.research.google.com/drive/1GmRiwmUH8E8KJ4d1g8vDVMGSdfmFFjd5)
3. Домашняя работа

Используя знания из [первой части](https://colab.research.google.com/drive/1tqXt0XNstMb2TeJ8Za9S2xQA6ipjyMPk) урока, а также базы знаний, полученной в [практической части](https://colab.research.google.com/drive/1GmRiwmUH8E8KJ4d1g8vDVMGSdfmFFjd5) урока, создайте телеграм-бот, который будет отвечать на вопросы из вашей базы знаний. Для создания телеграм-бота используйте библиотеку aiogram3. При отправке Telegram-боту команды `/help` он должен возвращать информацию о базе знаний: тематика, число записей в базе знаний, пример запроса к базе. Задание выполните в Блокноте, для этого вам необходимо вспомнить, как запустить асинхронный цикл в Google Colab.




In [ ]:
# Отключим предупреждения в колабе. Будет меньше лишней информации в выводе
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd  # В DataFrame будем хранить базу знаний и результат токинизации базы знаний

In [ ]:
!pip install aiogram

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
!pip install openai

In [ ]:
pip install tiktoken

In [ ]:
pip install --upgrade openai

In [ ]:
import pandas as pd
import ast
embeddings_path = "./winter_olympics_2022.csv"

df = pd.read_csv(embeddings_path)

# Конвертируем наши эмбединги из строк в списки
df['embedding'] = df['embedding'].apply(ast.literal_eval)

In [ ]:
import asyncio
import logging
import pandas as pd
from aiogram import Bot, Dispatcher, types
from aiogram.filters.command import Command
from openai import OpenAI
import os
import getpass
import tiktoken

# Включаем логирование, чтобы не пропустить важные сообщения
logging.basicConfig(level=logging.INFO)

# Замените "YOUR_BOT_TOKEN" на токен, который Вы получили от BotFather
API_TOKEN = '7939035635:AAG-k9eSrly9K0jX7yZT_2Dxuirfa6oXNS0'

# Объект бота
bot = Bot(token=API_TOKEN)
# Диспетчер
dp = Dispatcher()

# Установка API ключа
# Ключ sk-proj-LXWPtW6ry8SHTYZGi9MjT3BlbkFJhx4g9ZGo7JhVyfKhQy8u
os.environ["OPENAI_API_KEY"] = getpass.getpass("Введите OPENAI API Key:")

# Выбираем модель GPT
GPT_MODEL = "gpt-3.5-turbo"

# Создаём объект OpenAI
openai = OpenAI(
  api_key = os.environ.get("OPENAI_API_KEY"),
)
from scipy import spatial  # вычисляет сходство векторов
EMBEDDING_MODEL = "text-embedding-ada-002"

# Функция поиска
def strings_ranked_by_relatedness(
    query: str, # пользовательский запрос
    df: pd.DataFrame, # DataFrame со столбцами text и embedding (база знаний)
    relatedness_fn=lambda x, y: 1 - spatial.distance.cosine(x, y), # функция схожести, косинусное расстояние
    top_n: int = 100 # выбор лучших n-результатов
) -> tuple[list[str], list[float]]: # Функция возвращает кортеж двух списков, первый содержит строки, второй - числа с плавающей запятой
    """Возвращает строки и схожести, отсортированные от большего к меньшему"""

    # Отправляем в OpenAI API пользовательский запрос для токенизации
    query_embedding_response = openai.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query,
    )

    # Получен токенизированный пользовательский запрос
    query_embedding = query_embedding_response.data[0].embedding

    # Сравниваем пользовательский запрос с каждой токенизированной строкой DataFrame
    strings_and_relatednesses = [
        (row["text"], relatedness_fn(query_embedding, row["embedding"]))
        for i, row in df.iterrows()
    ]

    # Сортируем по убыванию схожести полученный список
    strings_and_relatednesses.sort(key=lambda x: x[1], reverse=True)

    # Преобразовываем наш список в кортеж из списков
    strings, relatednesses = zip(*strings_and_relatednesses)

    # Возвращаем n лучших результатов
    return strings[:top_n], relatednesses[:top_n]

# Функция для подсчета токенов
def num_tokens(text: str, model: str = GPT_MODEL) -> int:
    """Возвращает число токенов в строке для заданной модели"""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

# Функция формирования запроса к chatGPT по пользовательскому вопросу и базе знаний
def query_message(query: str, df: pd.DataFrame, model: str, token_budget: int) -> str:
    """Возвращает сообщение для GPT с соответствующими исходными текстами, извлеченными из фрейма данных (базы знаний)."""
    strings, relatednesses = strings_ranked_by_relatedness(query, df)  # Функция ранжирования базы знаний по пользовательскому запросу
    message = 'Use the below articles on the 2022 Winter Olympics to answer the subsequent question. If the answer cannot be found in the articles, write "I could not find an answer."'
    question = f"\n\nQuestion: {query}"

    for string in strings:
        next_article = f'\n\nWikipedia article section:\n"""\n{string}\n"""'
        if (num_tokens(message + next_article + question, model=model) > token_budget):
            break
        else:
            message += next_article
    return message + question

def ask(query: str, df: pd.DataFrame = df, model: str = GPT_MODEL, token_budget: int = 4096 - 500, print_message: bool = False) -> str:
    """Отвечает на вопрос, используя GPT и базу знаний."""
    message = query_message(query, df, model=model, token_budget=token_budget)
    if print_message:
        print(message)
    messages = [
        {"role": "system", "content": "You answer questions about the 2022 Winter Olympics."},
        {"role": "user", "content": message},
    ]
    response =  openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    response_message = response.choices[0].message.content
    return response_message

# Обработчик команды /start
@dp.message(Command("start"))
async def send_welcome(message: types.Message):
    welcome_message = (
        "Добро пожаловать в бот, посвященный Зимним Олимпийским играм 2022!\n"
        "Здесь Вы можете задавать вопросы и получать информацию из нашей базы знаний.\n"
        "Используйте команду /help для получения справки."
    )
    await message.answer(welcome_message)

# Обработчик команды /help
@dp.message(Command("help"))
async def send_help(message: types.Message):
    topic = "Зимние Олимпийские игры 2022"
    record_count = len(df)
    example_query = "Пример запроса: 'Что такое Зимние Олимпийские игры?'"
    help_message = (
        f"Тематика: {topic}\n"
        f"Число записей в базе знаний: {record_count}\n"
        f"{example_query}"
    )
    await message.answer(help_message)

# Обработчик текстовых сообщений
@dp.message()
async def handle_query(message: types.Message):
    user_query = message.text  # Получаем текст запроса от пользователя
    response = ask(user_query)  # Получаем ответ от функции ask
    await message.answer(response)  # Отправляем ответ пользователю

# Обработчик для несуществующих команд
@dp.message()
async def unknown_command(message: types.Message):
    await message.answer("Извините, я не понимаю эту команду.")

# Запуск процесса поллинга новых апдейтов
async def main():
    await dp.start_polling(bot)

if __name__ == "__main__":
    asyncio.run(main())


Введите OPENAI API Key:··········
